In [0]:
dbutils.fs.ls("/")

In [0]:
display(dbutils.fs.ls("/"))

In [0]:
display(dbutils.fs.ls("/databricks-datasets/"))

In [0]:
display(dbutils.fs.ls("/databricks-datasets/airlines/"))

In [0]:
df = spark.read.csv(
    "dbfs:/databricks-datasets/airlines/part-00000",
    header=True,
    inferSchema=True
)
display(df)

In [0]:
df1 = spark.read.csv(
    "dbfs:/databricks-datasets/airlines/part-00001",
    header=True,
    inferSchema=True
)
display(df1)

In [0]:
df.printSchema()

In [0]:
%sql
create database spark_sql.airlines

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

airlines_schema = StructType([
    StructField("year", IntegerType(), True),
    StructField("month", IntegerType(), True),
    StructField("day_of_month", IntegerType(), True),
    StructField("day_of_week", IntegerType(), True),
    StructField("dep_time", StringType(), True),
    StructField("crs_dep_time", IntegerType(), True),
    StructField("arr_time", StringType(), True),
    StructField("crs_arr_time", IntegerType(), True),
    StructField("unique_carrier", StringType(), True),
    StructField("flight_num", IntegerType(), True),
    StructField("tail_num", StringType(), True),
    StructField("actual_elapsed_time", StringType(), True),
    StructField("crs_elapsed_time", IntegerType(), True),
    StructField("air_time", StringType(), True),
    StructField("arr_delay", StringType(), True),
    StructField("dep_delay", StringType(), True),
    StructField("origin", StringType(), True),
    StructField("dest", StringType(), True),
    StructField("distance", StringType(), True),
    StructField("taxi_in", StringType(), True),
    StructField("taxi_out", StringType(), True),
    StructField("cancelled", IntegerType(), True),
    StructField("cancellation_code", StringType(), True),
    StructField("diverted", IntegerType(), True),
    StructField("carrier_delay", StringType(), True),
    StructField("weather_delay", StringType(), True),
    StructField("nas_delay", StringType(), True),
    StructField("security_delay", StringType(), True),
    StructField("late_aircraft_delay", StringType(), True),
    StructField("IsArrDelayed", StringType(), True),
    StructField("IsDepDelayed", StringType(), True),
    ])

In [0]:
airlines_df = (
    spark.read
        .schema(airlines_schema)                  # ✅ fixed schema
        .option("header", "false")                # your files have no header
        .csv("dbfs:/databricks-datasets/airlines/part-*")
)

display(airlines_df)

In [0]:
airlines_df.count()

In [0]:
airlines_df.printSchema()

#### Tempview Creation from Dataframe

In [0]:
airlines_df.createOrReplaceTempView("airlines_info")

In [0]:
%sql
select * from airlines_info

#### Source Table Creation from tempview

In [0]:
%sql
create table spark_sql.airlines.airlines_info
AS
select * from airlines_info

In [0]:
%sql
select * from spark_sql.airlines.airlines_info order by year desc

In [0]:
%sql
create database spark_sql.airlines_dwh

### 1. Staging
##### INITIAL LOAD

In [0]:
%sql
create table spark_sql.airlines_dwh.stg_airlines
as
select * from spark_sql.airlines.airlines_info where year = 1987

In [0]:
%sql
select * from spark_sql.airlines_dwh.stg_airlines

#### Transformation

In [0]:
%sql
select count(*) from spark_sql.airlines_dwh.stg_airlines 
where flight_num is not null
  and origin is not null
  and dest is not null
  and distance is not null
  and year is not null
  and month is not null
  and day_of_month is not null
  and day_of_week is not null;

In [0]:
%sql
select * from spark_sql.airlines_dwh.stg_airlines 
where cancelled = 1 
  and flight_num is not null
  and origin is not null
  and dest is not null
  and distance is not null
  and year is not null
  and month is not null
  and day_of_month is not null
  and day_of_week is not null;

In [0]:
%sql
create or replace view spark_sql.airlines_dwh.v_stg_airlines
as
select * from spark_sql.airlines_dwh.stg_airlines
where flight_num is not null
  and origin is not null
  and dest is not null
  and distance is not null
  and year is not null
  and month is not null
  and day_of_month is not null
  and day_of_week is not null;

#### Core Layer

In [0]:
%sql
create table spark_sql.airlines_dwh.airlines
as
select * from spark_sql.airlines_dwh.v_stg_airlines;

#### INCREMENTAL LOAD

In [0]:
%sql
create OR replace table spark_sql.airlines_dwh.stg_airlines
as
select * from spark_sql.airlines.airlines_info where year between 1988 AND 1990

#### Validating initial Load data

In [0]:
%sql
select * from spark_sql.airlines_dwh.stg_airlines where year = 1987

In [0]:
%sql
select * from spark_sql.airlines_dwh.stg_airlines

In [0]:
%sql
select distinct(year) from spark_sql.airlines_dwh.stg_airlines

#### Transformation Applied

In [0]:
%sql
create or replace view spark_sql.airlines_dwh.v_stg_airlines
as
select * from spark_sql.airlines_dwh.stg_airlines
where flight_num is not null
  and origin is not null
  and dest is not null
  and distance is not null
  and year is not null
  and month is not null
  and day_of_month is not null
  and day_of_week is not null;

#### Core Layer

In [0]:
%sql
insert into spark_sql.airlines_dwh.airlines
select * from spark_sql.airlines_dwh.v_stg_airlines;

In [0]:
%sql
select * from spark_sql.airlines_dwh.airlines order by year;

#### Incremental Load

In [0]:
%sql
create OR replace table spark_sql.airlines_dwh.stg_airlines
as
select * from spark_sql.airlines.airlines_info where year between 1990 AND 1995

In [0]:
%sql
select distinct(year) from spark_sql.airlines_dwh.stg_airlines

In [0]:
%sql
create or replace view spark_sql.airlines_dwh.v_stg_airlines
as
select * from spark_sql.airlines_dwh.stg_airlines
where flight_num is not null
  and origin is not null
  and dest is not null
  and distance is not null
  and year is not null
  and month is not null
  and day_of_month is not null
  and day_of_week is not null;

In [0]:
%sql
insert into spark_sql.airlines_dwh.airlines
select * from spark_sql.airlines_dwh.v_stg_airlines;

In [0]:
%sql
select * from spark_sql.airlines_dwh.airlines order by year desc;

##### Incremental Load by max of Year last loaded

In [0]:
%sql    
create OR replace table spark_sql.airlines_dwh.stg_airlines
as
select * from spark_sql.airlines.airlines_info 
where year > (select max(year) as loaded_year from spark_sql.airlines_dwh.airlines)

In [0]:
%sql
select distinct(year) from spark_sql.airlines_dwh.stg_airlines order by year desc

In [0]:
%sql
create or replace view spark_sql.airlines_dwh.v_stg_airlines
as
select * from spark_sql.airlines_dwh.stg_airlines
where flight_num is not null
  and origin is not null
  and dest is not null
  and distance is not null
  and year is not null
  and month is not null
  and day_of_month is not null
  and day_of_week is not null;

In [0]:
%sql
insert into spark_sql.airlines_dwh.airlines
select * from spark_sql.airlines_dwh.v_stg_airlines;

In [0]:
%sql  
select * from spark_sql.airlines_dwh.airlines order by year desc